# ==============================================================================
# 🧱 MATERIALS SCIENCE: CONCRETE STRENGTH VIA PROPER GRADIENT BOOSTING
# ==============================================================================
### Core Theoretical Principles:
Concrete compressive strength (MPa) depends non-linearly on 8 chemical components.
Gradient Boosting combines shallow trees ($d=3$) that capture complex 3-way chemical hydration kinetics without overfitting.


In [1]:
# STEP 1: ENTERPRISE ENVIRONMENT SETUP
import os
import time
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, KFold, GridSearchCV
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

np.random.seed(42)
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
print("✅ STEP 1: Concrete Materials Environment ready.")


✅ STEP 1: Concrete Materials Environment ready.


## 📥 STEP 2 & 3: INGESTION & DATASET HARMONIZATION
Loading Kaggle Concrete Compressive Strength dataset (1,030 mix formulations).


In [2]:
# STEP 2 & 3: LOAD & HARMONIZE
data_path = 'data/concrete/Concrete Compressive Strength.csv'
df = pd.read_csv(data_path)

df.columns = [c.strip().lower().replace(' ', '_').replace('(', '').replace(')', '') for c in df.columns]
target_col = [c for c in df.columns if 'strength' in c][0]

print(f"📊 Dataset Shape: {df.shape[0]} mixtures, {df.shape[1]} physical attributes")
print(f"Target Strength Summary:\n{df[target_col].describe().round(2)}")
df.head(3)


📊 Dataset Shape: 1030 mixtures, 9 physical attributes
Target Strength Summary:
count    1030.00
mean       35.82
std        16.71
min         2.33
25%        23.71
50%        34.44
75%        46.14
max        82.60
Name: concrete_compressive_strength, dtype: float64


,cement,blast_furnace_slag,fly_ash,water,superplasticizer,coarse_aggregate,fine_aggregate,age_day,concrete_compressive_strength
0,540.0,0.0,0.0,162.0,2.5,1040.0,676.0,28,79.986111
1,540.0,0.0,0.0,162.0,2.5,1055.0,676.0,28,61.887366
2,332.5,142.5,0.0,228.0,0.0,932.0,594.0,270,40.269535


## 🧹 STEP 4 & 5: SEGREGATION & HYGIENE
Separating features and target compressive strength.


In [3]:
# STEP 4 & 5: SEGREGATION
X = df.drop(columns=[target_col])
y = df[target_col]

print(f"Components: {X.shape[1]}")
print(f"Missing attributes: {X.isnull().sum().sum()}")


Components: 8
Missing attributes: 0


## 🔒 STEP 6: TRAIN-TEST SPLIT
80/20 train/test split.


In [4]:
# STEP 6: SPLIT
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)
print(f"Train Formulations: {X_train.shape[0]}")
print(f"Test Formulations : {X_test.shape[0]}")


Train Formulations: 824
Test Formulations : 206


## ⚠️ STEP 7 & 8: SINGLE TREE VS GRADIENT BOOSTING DUEL
Contrasting a single decision tree against Gradient Tree Boosting.


In [5]:
# STEP 7 & 8: ALGORITHM DUEL
pipe_single = DecisionTreeRegressor(max_depth=4, random_state=42)
pipe_single.fit(X_train, y_train)

pipe_gbr = GradientBoostingRegressor(n_estimators=100, learning_rate=0.1, max_depth=4, random_state=42)
pipe_gbr.fit(X_train, y_train)

r2_tree = r2_score(y_test, pipe_single.predict(X_test))
r2_gbr = r2_score(y_test, pipe_gbr.predict(X_test))

print("="*65)
print(f"Single Decision Tree R²   : {r2_tree*100:.2f}% | MAE: {mean_absolute_error(y_test, pipe_single.predict(X_test)):.2f} MPa")
print(f"Gradient Boosting Reg R²  : {r2_gbr*100:.2f}% | MAE: {mean_absolute_error(y_test, pipe_gbr.predict(X_test)):.2f} MPa")
print(f"Gradient Boosting Advance : +{(r2_gbr - r2_tree)*100:.2f}% R² gain!")
print("="*65)


Single Decision Tree R²   : 56.96% | MAE: 8.03 MPa
Gradient Boosting Reg R²  : 91.51% | MAE: 3.35 MPa
Gradient Boosting Advance : +34.55% R² gain!


## 🎯 STEP 9: TUNING HYPERPARAMETERS VIA 5-FOLD CV
Optimizing `learning_rate`, `subsample`, and `max_depth`.


In [6]:
# STEP 9: GRID SEARCH
param_grid = {
    'n_estimators': [100, 150],
    'learning_rate': [0.05, 0.1],
    'max_depth': [3, 4],
    'subsample': [0.8, 1.0]
}

cv = KFold(n_splits=5, shuffle=True, random_state=42)
grid = GridSearchCV(pipe_gbr, param_grid=param_grid, cv=cv, scoring='r2', n_jobs=-1)
grid.fit(X_train, y_train)

best_model = grid.best_estimator_
print(f"🏆 Best Concrete Hyperparameters: {grid.best_params_}")
print(f"🎯 Best 5-Fold CV R²: {grid.best_score_*100:.2f}%")


🏆 Best Concrete Hyperparameters: {'learning_rate': 0.1, 'max_depth': 4, 'n_estimators': 150, 'subsample': 0.8}
🎯 Best 5-Fold CV R²: 91.63%


## 📊 STEP 10 & 11: SPECIFICATION METRICS & RESIDUALS
Evaluating compressive load tolerances.


In [7]:
# STEP 10 & 11: EVALUATION REPORT
y_pred = best_model.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print("📋 CIVIL ENGINEERING QUALITY REPORT:")
print(f"   Mean Absolute Error (MAE) : {mae:.2f} MPa")
print(f"   Root Mean Squared Error   : {rmse:.2f} MPa")
print(f"   R² Coefficient            : {r2 * 100:.2f}%")


📋 CIVIL ENGINEERING QUALITY REPORT:
   Mean Absolute Error (MAE) : 3.10 MPa
   Root Mean Squared Error   : 4.51 MPa
   R² Coefficient            : 92.11%


## 💾 STEP 12 & 13: PRODUCTION SERIALIZATION & BATCH SMOKE TEST
Deploying pipeline and testing batch plant curing quality check latency.


In [8]:
# STEP 12 & 13: SERIALIZATION & SMOKE TEST
os.makedirs('production_models', exist_ok=True)
model_path = 'production_models/concrete_gbr_pipeline.joblib'
joblib.dump(best_model, model_path, compress=3)
print(f"💾 Production pipeline saved to: {model_path} ({os.path.getsize(model_path):,} bytes)")

loaded_pipe = joblib.load(model_path)
sample_mix = X_test.iloc[[0]]

t0 = time.perf_counter()
pred_strength = loaded_pipe.predict(sample_mix)[0]
latency_ms = (time.perf_counter() - t0) * 1000

print(f"\n🏗️ LIVE BATCH PLANT MIX AUDIT:")
print(f"   Forecast Compressive Strength: 🧱 {pred_strength:.2f} MPa")
print(f"   Actual Lab Test Strength     : {y_test.iloc[0]:.2f} MPa")
print(f"   Batch Verification Latency   : {latency_ms:.3f} ms (SLA < 2.0ms: {'PASS' if latency_ms < 2.0 else 'CHECK'})")


💾 Production pipeline saved to: production_models/concrete_gbr_pipeline.joblib (110,669 bytes)

🏗️ LIVE BATCH PLANT MIX AUDIT:
   Forecast Compressive Strength: 🧱 51.49 MPa
   Actual Lab Test Strength     : 52.91 MPa
   Batch Verification Latency   : 1.506 ms (SLA < 2.0ms: PASS)
